In [ ]:
import random
import json
import re
import os
import base64
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from local_variables import PAINTING_STYLES

In [ ]:
system_prompt = """You are an expert at generating realistic paintings for a variety of art styles."""
user_prompt_template = """Generate an image of a non-existing painting of style {painting_style}"""


In [ ]:
model_name = "gpt-5-mini"
model_kwargs = {}
model_kwargs["reasoning_effort"] = "low"
model_kwargs["service_tier"] = "flex" 
model_kwargs["tools"] = [{"type": "image_generation"}]
# kwargs["temperature"] = 1
painting_style = "abstract art"
user_prompt = user_prompt_template.format(painting_style=painting_style)
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
image_data = make_llm_request(model_name, messages, **model_kwargs)
image_data
if image_data:
    image_base64 = image_data[0]
    with open("./data/test1.png", "wb") as f:
        f.write(base64.b64decode(image_base64))

In [ ]:
models = ["gpt-5-mini"]
# models = ["gpt-5"]
model_kwargs = {}
model_kwargs["reasoning_effort"] = "low"
model_kwargs["tools"] = [{"type": "image_generation"}]
model_kwargs["service_tier"] = "flex" 
# kwargs["temperature"] = 1

async def generate_paintings(n, system_prompt=system_prompt, user_prompt_template=user_prompt_template, base_path_to_save_images="./data"): 
    tasks = []
    for model_name in models:
        print(f"Generating painting with {model_name}...")
        for painting_style in PAINTING_STYLES:
            for i in range(n):
                print(f"Queueing painting about {painting_style}")
                user_prompt = user_prompt_template.format(painting_style=painting_style)
                # Instead of awaiting here, append coroutine to tasks
                payload = {
                    "model": model_name,
                    "system_prompt": system_prompt,
                    "user_prompt": user_prompt,
                    "painting_style": painting_style,
                    "index": i,
                }
                messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
                tasks.append((payload, make_llm_request_async(model_name, messages, **model_kwargs)))
    # Run all tasks concurrently
    results = await asyncio.gather(*[t[-1] for t in tasks], return_exceptions=True)
    payloads = []
    for idx, (payload, _) in enumerate(tasks):
        image_data = results[idx]
        painting_style = payload["painting_style"]
        index = payload["index"]
        payloads.append(payload)
        if image_data:
            image_base64 = image_data[0]
            files = os.listdir(f"{base_path_to_save_images}/{painting_style}") if os.path.exists(f"{base_path_to_save_images}/{painting_style}") else []
            existing_indices = [int(re.match(r"(\d+)\.png", fname).group(1)) for fname in files if re.match(r"(\d+)\.png", fname)]
            if existing_indices:
                index = max(existing_indices) + 1
            else:
                index = 0
            image_path = f"{base_path_to_save_images}/{painting_style}/{index}.png"
            if not os.path.exists(os.path.dirname(image_path)):
                os.makedirs(os.path.dirname(image_path))
            with open(image_path, "wb") as f:
                f.write(base64.b64decode(image_base64))
    return payloads

# PAINTING_STYLES = ["expressionism"]

# run the async function
n = 1 # number of paintings to generate per painting_style
base_path_to_save_images="./data"
payloads = await generate_paintings(n, system_prompt, user_prompt_template, base_path_to_save_images=base_path_to_save_images)
print(f"Generated {len(payloads)} paintings and saved to {base_path_to_save_images}")